## Dependencies

In [1]:
import pathlib
import numpy as np
import pandas as pd

## Download & Create EC -> GO Term Mapping

In [4]:
!curl -s -L -o ec2go.txt http://www.geneontology.org/external2go/ec2go

with open('ec2go.txt', 'r') as file:
    data = file.readlines()

# A dictionary to store InterPro codes and their associated GO terms
# If there are multiple GO terms for each interpro entry, they will be stored in an array
ec2go = {}

# Extract InterPro code and GO code for a given line
def extract_codes(line):
    parts = line.strip().split(' ')
    ec_code = parts[0].split(':')[1]
    go_code = parts[-1]
    return ec_code, go_code

for line in data:
    if line[0] == '!':
        continue
    ec_code, go_code = extract_codes(line)
   
    if ec_code in ec2go:
        ec2go[ec_code].append(go_code)
    else:
        ec2go[ec_code] = [go_code]

## Download & Create dataset

In [5]:
flat = "https://www.ebi.ac.uk/thornton-srv/m-csa/media/flat_files/curated_data.csv"
!curl -s -O $flat

In [6]:
curated_df = pd.read_csv("curated_data.csv", on_bad_lines="skip")

In [7]:
def GO_mapping(dataframe, ec2go=ec2go):
    output_go = []
    output_nogo = []
    proteins = np.unique(dataframe["Uniprot IDs"].values)

    for protein in proteins:
        new_df = dataframe[dataframe["Uniprot IDs"] == protein].copy()

        indices = np.unique(new_df["resid/chebi id"].values) 
        indices = indices.astype(int).tolist()
        
        
        enzyme_class = new_df["EC"].values[0]

        try:
            go_term = ec2go[enzyme_class]
            output_go.append((protein, indices, enzyme_class, go_term))
            # print(f"Protein: {protein}, Indices: {indices}, Enzyme Class: {enzyme_class}, GO Term: {go_term}")
        except KeyError:
            output_nogo.append((protein, indices, enzyme_class))
            # print(f"Protein: {protein}, Indices: {indices}, Enzyme Class: {enzyme_class}")
            
    return output_go, output_nogo

dataset, _ = GO_mapping(curated_df)
        

In [20]:
import pandas as pd
dataset
# Create a dataframe from the dataset
dataset_df = pd.DataFrame(dataset, columns=['UniprotID', 'AnnotatedIndices', 'EnzymeClass', 'GOTerm'])
dataset_df.to_csv('dataset.csv', index=False)

dataset_df.head()

,UniprotID,AnnotatedIndices,EnzymeClass,GOTerm
0,A0A0F6FBL7,"[14, 83, 197]",5.1.1.13,[GO:0047689]
1,A0QTN8,"[162, 191, 217, 242, 266, 294, 18420]",5.5.1.1,[GO:0018849]
2,A2RJT9,"[43, 130, 57618]",1.3.98.1,[GO:1990663]
3,A4XF23,"[147, 159, 210, 212, 236, 262, 283, 339, 402, ...",4.2.1.8,[GO:0008927]
4,A5JTM5,"[64, 86, 90, 114, 137, 145]",3.8.1.7,[GO:0018787]


## Get sequence and do features/labels for given go term

In [54]:
import requests as r
from Bio import SeqIO
from io import StringIO

def uniprot2seq(ID: str):
    baseUrl="http://www.uniprot.org/uniprot/"
    currentUrl=baseUrl+ID+".fasta"
    response = r.post(currentUrl)
    cData=''.join(response.text)

    Seq=StringIO(cData)
    pSeq=list(SeqIO.parse(Seq,'fasta'))
    try:
        print(f"Sequence for {ID} found! Returning...{str(pSeq[0].seq)}")
        return str(pSeq[0].seq)
    except IndexError:
        print(f"Sequence for {ID} not found! Returning...None")
        return None

,UniprotID,AnnotatedIndices,EnzymeClass,GOTerm
0,A0A0F6FBL7,"[14, 83, 197]",5.1.1.13,[GO:0047689]
1,A0QTN8,"[162, 191, 217, 242, 266, 294, 18420]",5.5.1.1,[GO:0018849]
2,A2RJT9,"[43, 130, 57618]",1.3.98.1,[GO:1990663]
3,A4XF23,"[147, 159, 210, 212, 236, 262, 283, 339, 402, ...",4.2.1.8,[GO:0008927]
4,A5JTM5,"[64, 86, 90, 114, 137, 145]",3.8.1.7,[GO:0018787]


In [57]:
dataset_df['Sequence'] = dataset_df['UniprotID'].apply(uniprot2seq)

Sequence for A0A0F6FBL7 not found! Returning...None
Sequence for A0QTN8 found! Returning...MKIVAIGAIPFSIPYTKPLRFASGEVHAAEHVLVRVHTDDGIVGVAEAPPRPFTYGETQTGIVAVIEQYFAPALIGLTLTEREVAHTRMARTVGNPTAKAAIDMAMWDALGQSLRLSVSEMLGGYTDRMRVSHMLGFDDPVKMVAEAERIRETYGINTFKVKVGRRPVQLDTAVVRALRERFGDAIELYVDGNRGWSAAESLRAMREMADLDLLFAEELCPADDVLSRRRLVGQLDMPFIADESVPTPADVTREVLGGSATAISIKTARTGFTGSTRVHHLAEGLGLDMVMGNQIDGQIGTACTVSFGTAFERTSRHAGELSNFLDMSDDLLTVPLQISDGQLHRRPGPGLGIEIDPDKLAHYRTDN
Sequence for A2RJT9 found! Returning...MLNTTFANAKFANPFMNASGVHCMTIEDLEELKASQAGAYITKSSTLEKREGNPLPRYVDLELGSINSMGLPNLGFDYYLDYVLKNQKENAQEGPIFFSIAGMSAAENIAMLKKIQESDFSGITELNLSCPNVPGKPQLAYDFEATEKLLKEVFTFFTKPLGVKLPPYFDLVHFDIMAEILNQFPLTYVNSVNSIGNGLFIDPEAESVVIKPKDGFGGIGGAYIKPTALANVRAFYTRLKPEIQIIGTGGIETGQDAFEHLLCGATMLQIGTALHKEGPAIFDRIIKELEEIMNQKGYQSIADFHGKLKSL
Sequence for A4XF23 found! Returning...MKITAARVIITCPGRNFVTLKIETDQGVYGIGDATLNGRELSVVAYLQEHVAPCLIGMDPRRIEDIWQYVYRGAYWRRGPVTMRAIAAVDMALWDIKAKMAGMPLYQLLGGRSRDGIMVYGHANGSDIAETVEAVGHYIDMGYKAIRAQTG

In [ ]:
dataset_df.to_csv('dataset_seq.csv', index=False)
dataset_df.tail()

,UniprotID,AnnotatedIndices,EnzymeClass,GOTerm,Sequence
842,Q9YHT1,"[130, 251, 253, 263, 266, 297, 364, 408, 33737...",1.3.5.1,[GO:0008177],MAAVVAASRSLAKCWLRPAVRAWPAACQTHARNFHFTVDGKKNAST...
843,Q9ZAG3,"[53, 55, 99, 101, 132]",3.3.2.8,[GO:0018744],MTSKIEQPRWASKDSAAGAASTPDEKIVLEFMDALTSNDAAKLIEY...
844,Q9ZF13,"[50, 127, 128, 196, 198, 225, 254]",3.2.1.78,[GO:0016985],GLHVKNGRLYEANGQEFIIRGVSHPHNWYPQHTQAFADIKSHGANT...
845,Q9ZHI0,"[114, 116, 118, 221, 312, 406, 409, 422, 427]",6.5.1.2,[GO:0003911],MTREEARRRINELRDLIRYHNYRYYVLADPEISDAEYDRLLRELKE...
846,Q9ZP19,"[88, 109, 118, 240, 244, 261, 274, 47292]",1.10.3.1,[GO:0004097],APIQAPEISKCVVPPADLPPGAVVDNCCPPVASNIVDYKLPAVTTM...
